# 02 — Prétraitement des nuages de points

Ce notebook illustre le pipeline de données du mini-projet :
- chargement des fichiers `.ply`
- sous-échantillonnage à un nombre fixe de points
- normalisation (sphère unité)
- augmentations (entraînement uniquement)
- construction des splits `train/val/test` et partitions fédérées (IID / non-IID)

Les fonctions réutilisées proviennent de `src/feastnet_fl/data/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from plyfile import PlyData

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from feastnet_fl.config import load_config
from feastnet_fl.data.prepare import prepare_experiment_splits
from feastnet_fl.data.transforms import (
    PointCloudAugmentation,
    normalize_points,
    sample_or_pad_points,
)

CONFIG_PATH = PROJECT_ROOT / "configs" / "feastnet_ultra_quick_cpu.yaml"
config = load_config(str(CONFIG_PATH))
config["project_root"] = str(PROJECT_ROOT)

METADATA_CSV = PROJECT_ROOT / config["paths"]["metadata_csv"]
POINTCLOUD_ROOT = PROJECT_ROOT / config["paths"]["pointcloud_root"]
SPLIT_ROOT = PROJECT_ROOT / config["paths"]["split_root"]
NUM_POINTS = config["sampling"]["num_points"]
SEED = config["seed"]

sns.set_theme(style="whitegrid")
print(f"Projet : {PROJECT_ROOT}")
print(f"Métadonnées : {METADATA_CSV}")
print(f"Points cibles : {NUM_POINTS}")

## 1. Exploration du sous-ensemble

In [ ]:
metadata = pd.read_csv(METADATA_CSV)
print(f"Nombre d'objets : {len(metadata)}")
print(f"Nombre de classes : {metadata['label'].nunique()}")
metadata.head()

In [ ]:
class_counts = metadata["label"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
class_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Distribution des classes dans le sous-ensemble")
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'objets")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

print(f"Min / max / moyenne par classe : {class_counts.min()} / {class_counts.max()} / {class_counts.mean():.1f}")

## 2. Chargement brut d'un fichier `.ply`

In [ ]:
def load_raw_ply(object_id: str, pointcloud_root: Path):
    candidates = [
        pointcloud_root / f"{object_id}.ply",
        pointcloud_root / object_id.split("_")[0] / f"{object_id}.ply",
    ]
    ply_path = next(path for path in candidates if path.exists())
    vertex = PlyData.read(str(ply_path))["vertex"]
    names = vertex.data.dtype.names
    pos = np.stack([vertex[axis] for axis in ("x", "y", "z")], axis=1).astype(np.float32)
    if all(channel in names for channel in ("red", "green", "blue")):
        colors = np.stack([vertex[channel] for channel in ("red", "green", "blue")], axis=1) / 255.0
    else:
        colors = np.ones_like(pos) * 0.5
    return pos, colors, ply_path


def plot_point_cloud(pos, colors, title="", ax=None, point_size=1.0):
    if ax is None:
        fig = plt.figure(figsize=(5, 5))
        ax = fig.add_subplot(111, projection="3d")
    ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=colors, s=point_size)
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    return ax


sample_row = metadata.sample(1, random_state=SEED).iloc[0]
raw_pos, raw_colors, ply_path = load_raw_ply(sample_row["id"], POINTCLOUD_ROOT)
print(f"Objet : {sample_row['id']} | Classe : {sample_row['label']}")
print(f"Fichier : {ply_path}")
print(f"Points bruts : {len(raw_pos)}")

plot_point_cloud(raw_pos, raw_colors, title=f"Brut — {sample_row['label']}")
plt.tight_layout()
plt.show()

## 3. Sous-échantillonnage (`sample_or_pad_points`)

In [ ]:
torch.manual_seed(SEED)
pos_tensor = torch.from_numpy(raw_pos)
x_tensor = torch.from_numpy(np.concatenate([raw_pos, raw_colors], axis=1))

sampled_pos, sampled_x = sample_or_pad_points(pos_tensor, x_tensor, NUM_POINTS)
sampled_colors = sampled_x[:, 3:].numpy()

fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")
plot_point_cloud(raw_pos, raw_colors, title=f"Avant ({len(raw_pos)} pts)", ax=ax1)
plot_point_cloud(sampled_pos.numpy(), sampled_colors, title=f"Après ({NUM_POINTS} pts)", ax=ax2)
plt.tight_layout()
plt.show()

## 4. Normalisation (centrage + mise à l'échelle unitaire)

In [ ]:
normalized_pos = normalize_points(sampled_pos)

fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")
plot_point_cloud(sampled_pos.numpy(), sampled_colors, title="Avant normalisation", ax=ax1)
plot_point_cloud(normalized_pos.numpy(), sampled_colors, title="Après normalisation", ax=ax2)
plt.tight_layout()
plt.show()

print(
    "Rayon max après normalisation :",
    float(normalized_pos.norm(dim=1).max()),
)

## 5. Augmentations (rotation, scale, jitter)

In [ ]:
augmentation = PointCloudAugmentation()
base_pos = normalized_pos.clone()
base_x = torch.cat([base_pos, sampled_x[:, 3:]], dim=1)

fig = plt.figure(figsize=(14, 4))
titles = ["Original", "Augmentation 1", "Augmentation 2", "Augmentation 3"]
for idx, title in enumerate(titles):
    ax = fig.add_subplot(1, 4, idx + 1, projection="3d")
    if idx == 0:
        display_pos = base_pos.numpy()
        display_colors = base_x[:, 3:].numpy()
    else:
        torch.manual_seed(SEED + idx)
        aug_pos, aug_x = augmentation(base_pos.clone(), base_x.clone())
        display_pos = aug_pos.numpy()
        display_colors = aug_x[:, 3:].numpy()
    plot_point_cloud(display_pos, display_colors, title=title, ax=ax, point_size=2.0)

plt.suptitle("Comparaison avant / après augmentations")
plt.tight_layout()
plt.show()

## 6. Splits train / validation / test

In [ ]:
if not (SPLIT_ROOT / "train.csv").exists():
    prepare_experiment_splits(config)

splits = {
    name: pd.read_csv(SPLIT_ROOT / f"{name}.csv")
    for name in ("train", "val", "test")
}

for name, split_df in splits.items():
    print(f"{name:5s} : {len(split_df):4d} objets | {split_df['label'].nunique()} classes")

split_counts = []
for split_name, split_df in splits.items():
    for label, count in split_df["label"].value_counts().items():
        split_counts.append({"split": split_name, "label": label, "count": count})
split_counts_df = pd.DataFrame(split_counts)

pivot = split_counts_df.pivot(index="label", columns="split", values="count").fillna(0)
pivot = pivot.reindex(columns=["train", "val", "test"])

plt.figure(figsize=(12, 5))
pivot.plot(kind="bar", stacked=False, figsize=(12, 5))
plt.title("Répartition par classe — train / val / test")
plt.xlabel("Classe")
plt.ylabel("Nombre d'objets")
plt.xticks(rotation=75, ha="right")
plt.legend(title="Split")
plt.tight_layout()
plt.show()

## 7. Partitions fédérées horizontales (IID vs non-IID)

In [ ]:
def load_client_splits(scenario: str):
    client_dir = SPLIT_ROOT / scenario
    client_files = sorted(client_dir.glob("*.csv"))
    return {path.stem: pd.read_csv(path) for path in client_files}


def plot_client_partition(client_splits: dict, title: str):
    rows = []
    for client_name, client_df in client_splits.items():
        for label, count in client_df["label"].value_counts().items():
            rows.append({"client": client_name, "label": label, "count": count})
    partition_df = pd.DataFrame(rows)
    pivot = partition_df.pivot(index="label", columns="client", values="count").fillna(0)

    plt.figure(figsize=(12, 5))
    pivot.plot(kind="bar", figsize=(12, 5))
    plt.title(title)
    plt.xlabel("Classe")
    plt.ylabel("Nombre d'objets")
    plt.xticks(rotation=75, ha="right")
    plt.tight_layout()
    plt.show()

    summary = partition_df.groupby("client")["count"].sum().rename("total_objects")
    print(summary)
    print("Classes par client :")
    for client_name, client_df in client_splits.items():
        print(f"  {client_name}: {client_df['label'].nunique()} classes")


iid_clients = load_client_splits("iid")
non_iid_clients = load_client_splits("non_iid")

plot_client_partition(iid_clients, "Partition IID — chaque client voit toutes les classes")
plot_client_partition(non_iid_clients, "Partition non-IID — classes par domaine sémantique (HFL)")

## 8. Pipeline complet via `PointCloudClassificationDataset`

Vérification que le dataset PyG applique bien toutes les étapes de prétraitement.

In [ ]:
from feastnet_fl.data.dataset import PointCloudClassificationDataset

dataset = PointCloudClassificationDataset(
    metadata_csv=str(METADATA_CSV),
    pointcloud_root=str(POINTCLOUD_ROOT),
    metadata=metadata.head(3),
    num_points=NUM_POINTS,
    normalize=True,
    augment=True,
)

for idx in range(len(dataset)):
    sample = dataset[idx]
    print(
        f"Sample {idx} | pos={tuple(sample.pos.shape)} | x={tuple(sample.x.shape)} | "
        f"label={sample.y.item()} | rayon max={sample.pos.norm(dim=1).max():.3f}"
    )